# 🚀 AI Designer Backend — Kaggle Edition

**Ưu điểm trên Kaggle:** GPU T4/P100 miễn phí, internet ổn định, session ~9h.

---

## Cách sử dụng:
1. Dataset `ai-gen-logo` đã upload sẵn ở `/kaggle/input/ai-gen-logo/`
2. **Cell 3** copy toàn bộ code ra `/kaggle/working/AI_GEN_IMAGE/`
3. **Cell 3b** verify LoRA outputs có đủ không
4. Chạy lần lượt: Cell 1 → 2 → 3 → 3b → 4 → 5b → **6**
5. **Cell 6** giữ chạy — copy ngrok URL để dùng ở frontend

---

### CELL 1: Cài Dependencies

In [ ]:
# THAY ĐỔI URL này bằng repo của bạn!
GIT_URL = 'https://github.com/My-Name-Hung/AI_GEN_IMGAE.git'  # @param {type:"string"}

!git clone $GIT_URL /kaggle/working/AI_GEN_IMAGE 2>&1 | tail -5
print('✅ Cloned')

# PATCH: Fix adapter_name bug in lora_manager.py
LM_FILE = '/kaggle/working/AI_GEN_IMAGE/app/services/lora_manager.py'
if os.path.exists(LM_FILE):
    with open(LM_FILE, 'r') as f:
        content = f.read()
    old = 'self.adapter_name = f"lora_{name}"'
    new = 'self.adapter_name = name'
    if old in content:
        content = content.replace(old, new)
        with open(LM_FILE, 'w') as f:
            f.write(content)
        print('✅ Fixed: adapter_name double-prefix bug')
    else:
        print('✅ No patch needed')
else:
    print('⚠️ lora_manager.py not found')

---

### CELL 2: Dataset info

Dataset `ai-gen-logo` đã mount ở `/kaggle/input/ai-gen-logo/`.
Chứa cả code (`app/`, `requirements.txt`, ...) và LoRA outputs.

In [ ]:
# Cài các package còn thiếu trên Kaggle
!pip install -q pyngrok ngrok python-multipart aiofiles pydantic-settings
!pip install -q opencv-python opencv-contrib-python
!pip install -q open-clip-torch peft safetensors scikit-image
!pip install -q huggingface_hub
print('✅ Dependencies installed')

---

### CELL 3: Copy code ra working directory

In [ ]:
import os
import getpass

HF_TOKEN = getpass.getpass('🔑 HuggingFace Token: ')
os.environ['HF_TOKEN'] = HF_TOKEN

NGROK_TOKEN = getpass.getpass('🔑 ngrok Authtoken (https://ngrok.com) : ')
os.environ['NGROK_TOKEN'] = NGROK_TOKEN

from pyngrok import conf
conf.get_default().auth_token = NGROK_TOKEN

print('✅ Tokens OK')

---

### CELL 3b: Verify LoRA outputs

In [ ]:
import os

PROJECT_DIR = '/kaggle/working/AI_GEN_IMAGE'
outputs_dir = f'{PROJECT_DIR}/outputs'

print('📦 LoRA adapters:')
if os.path.exists(outputs_dir):
    for lora in sorted(os.listdir(outputs_dir)):
        lora_path = os.path.join(outputs_dir, lora, 'final', 'unet_lora')
        has_safetensors = os.path.exists(os.path.join(lora_path, 'adapter_model.safetensors'))
        has_config = os.path.exists(os.path.join(lora_path, 'adapter_config.json'))
        size_mb = 0
        sf_path = os.path.join(lora_path, 'adapter_model.safetensors')
        if has_safetensors:
            size_mb = os.path.getsize(sf_path) / 1024 / 1024
        status = '✅' if (has_safetensors and has_config) else '❌'
        print(f'  {status} {lora}: {size_mb:.1f} MB | safetensors={has_safetensors}, config={has_config}')
else:
    print('  ⚠️ outputs/ không tìm thấy')

---

### CELL 4: Nhập Tokens (MỖI LẦN RECONNECT)

In [ ]:
import os

DATASET_DIR = '/kaggle/input/datasets/hng111120/ai-gen-logo'

print('📁 Dataset contents:')
for item in sorted(os.listdir(DATASET_DIR)):
    full = os.path.join(DATASET_DIR, item)
    if os.path.isdir(full):
        print(f'   📂 {item}/')
    else:
        print(f'   📄 {item}')
print()

# Verify key items
has_outputs = os.path.exists(os.path.join(DATASET_DIR, 'outputs'))
has_app = os.path.exists(os.path.join(DATASET_DIR, 'app'))
print(f'{"✅" if has_outputs else "❌"} outputs/: {"có" if has_outputs else "không"}')
print(f'{"✅" if has_app else "❌"} app/: {"có" if has_app else "không"}')

---

### CELL 5: Setup + Pre-download Model ⭐

Cell này:
1. Fix CORS
2. Verify GPU
3. **Pre-download SDXL Turbo** (~3-10 phút lần đầu)
4. Load LoRA adapters
5. Warmup generation

**CHỈ CHẠY 1 LẦN** — sau đó base model được cache.

In [ ]:
import os
import shutil

DATASET_DIR = '/kaggle/input/datasets/hng111120/ai-gen-logo'
PROJECT_DIR = '/kaggle/working/AI_GEN_IMAGE'

# Copy outputs/ từ dataset vào project (KHÔNG xóa project)
os.makedirs(PROJECT_DIR, exist_ok=True)

outputs_src = os.path.join(DATASET_DIR, 'outputs')
outputs_dst = os.path.join(PROJECT_DIR, 'outputs')

if os.path.exists(DATASET_DIR):
    if os.path.exists(outputs_src):
        if os.path.exists(outputs_dst):
            print('outputs/ da ton tai, ghi de...')
            shutil.rmtree(outputs_dst)
        shutil.copytree(outputs_src, outputs_dst)
        print(f'✅ Da copy outputs/ ({len(os.listdir(outputs_dst))} LoRA types)')
    else:
        print(f'⚠️ outputs/ khong ton tai trong dataset')
        print('   Dataset contents:', os.listdir(DATASET_DIR))
else:
    print(f'⚠️ Dataset chua duoc mount: {DATASET_DIR}')
    print('   CHAY CELL 2 TRUOC! (Add Data -> ai-gen-logo dataset)')

# Verify & fallback: tao dummy neu thieu (de test backend)
print()
print('LoRA files:')
for lt in ['lora_logo_2d', 'lora_logo_3d', 'lora_poster']:
    wt_dir = os.path.join(PROJECT_DIR, 'outputs', lt, 'final', 'unet_lora')
    wt = os.path.join(wt_dir, 'adapter_model.safetensors')
    if os.path.exists(wt):
        print(f'  ✅ {lt}: {os.path.getsize(wt)/1e6:.0f} MB')
    else:
        print(f'  ⚠️  {lt}: MISSING — can upload len Kaggle dataset')
        os.makedirs(wt_dir, exist_ok=True)

---

### CELL 5b: Pre-download SDXL Turbo + Load LoRA

**Thời gian:** ~3-10 phút lần đầu (download ~6GB). Lần sau chỉ ~10s.

In [ ]:
import os
import gc
import time
import torch

PROJECT_DIR = '/kaggle/working/AI_GEN_IMAGE'
os.environ['PYTHONPATH'] = PROJECT_DIR

import sys
if PROJECT_DIR not in sys.path:
    sys.path.insert(0, PROJECT_DIR)

os.environ['LORA_OUTPUTS_DIR'] = PROJECT_DIR + '/outputs'
print('LORA_OUTPUTS_DIR=' + os.environ['LORA_OUTPUTS_DIR'])

HF_TOKEN = os.environ.get('HF_TOKEN', '')

t0 = time.time()

try:
    # Use importlib.util to load from file path directly — bypasses sys.path issues
    import importlib.util

    lora_mgr_path = os.path.join(PROJECT_DIR, 'app', 'services', 'lora_manager.py')
    spec = importlib.util.spec_from_file_location('lora_manager_module', lora_mgr_path)
    lora_manager_mod = importlib.util.module_from_spec(spec)
    spec.loader.exec_module(lora_manager_mod)

    # Also remove any stale app.* imports from sys.modules so re-import picks up the fresh one
    for mod in list(sys.modules.keys()):
        if 'lora_manager' in mod or mod.startswith('app.'):
            del sys.modules[mod]

    sys.modules['app.services.lora_manager'] = lora_manager_mod
    _lm = lora_manager_mod
    _lm._lora_manager = None
    print('Module loaded from file: {}'.format(lora_mgr_path))
    print('Singleton reset OK')

    from diffusers import StableDiffusionPipeline, DPMSolverMultistepScheduler
    from app.services.lora_manager import get_lora_manager

    # Detect device BEFORE loading pipeline
    device = 'cuda' if torch.cuda.is_available() else 'cpu'
    dtype = torch.float16 if device == 'cuda' else torch.float32
    print('Device: {} | dtype: {}'.format(device, dtype))

    # Load base pipeline
    print('Downloading SDXL Turbo (~6GB)...')
    pipeline = StableDiffusionPipeline.from_pretrained(
        'stabilityai/sdxl-turbo',
        token=HF_TOKEN,
        variant='fp16' if device == 'cuda' else None,
        torch_dtype=dtype,
        safety_checker=None,
        requires_safety_checker=False,
    )
    print('Base model loaded')

    # Move pipeline to device
    pipeline = pipeline.to(device)
    print('Pipeline on {}'.format(device))

    # Load LoRA adapters
    lora_mgr = get_lora_manager()
    discovered = lora_mgr.available_types()
    print('LoRA Manager: {}'.format(discovered))
    if not discovered:
        print('   No LoRA found - using pure SDXL')
        print('   To use LoRA: upload outputs/ folder to Kaggle')
    else:
        for lora_type in discovered:
            ok = lora_mgr.load_adapter(lora_type, pipeline, scale=1.0, stack=False)
            status = 'OK' if ok else 'FAIL'
            print('   {} {}'.format(status, lora_type))
        if discovered:
            lora_mgr.unload_all(pipeline)

    # Apply scheduler + xformers
    if device == 'cuda':
        try:
            pipeline.enable_xformers_memory_efficient_attention()
            print('xformers enabled')
        except Exception as e:
            print('(xformers skip: {})'.format(e))
        pipeline.scheduler = DPMSolverMultistepScheduler.from_config(pipeline.scheduler.config)

    # Warmup
    print()
    print('Warmup GPU...')
    if device == 'cuda':
        torch.cuda.synchronize()
        print('GPU warmup done')
    print('Base model ready')

    # Cleanup
    del pipeline
    gc.collect()
    if device == 'cuda':
        torch.cuda.empty_cache()

    elapsed = time.time() - t0
    print()
    print('=' * 60)
    print('PRE-DOWNLOAD DONE! Time: {:.1f}s'.format(elapsed))
    print('   Base model cached - Cell 6 will start FAST')
    print('=' * 60)

except Exception as e:
    print('Error: {}'.format(e))
    import traceback
    traceback.print_exc()


---

### CELL 6: Khởi động Backend ⭐ (GIỮ CHẠY MÃI!)

Cell này in ra ngrok URL. COPY URL đó. ĐỪNG STOP cell.

In [ ]:
# ⭐ CELL 6: Start Backend + ngrok (FIXED)

import os
import subprocess
import time
import urllib.request
import json
from pyngrok import ngrok

PROJECT_DIR = '/kaggle/working/AI_GEN_IMAGE'
os.chdir(PROJECT_DIR)
os.environ['PYTHONPATH'] = PROJECT_DIR

# STEP 0: Remove any temp Python files to avoid uvicorn reload loop
subprocess.run(
    ['bash', '-c', 'rm -f ' + PROJECT_DIR + '/_ngrok_cell.py ' + PROJECT_DIR + '/_export_*.py ' + PROJECT_DIR + '/kaggle_cell*.py'],
    capture_output=True
)

# STEP 1: Kill old processes
print('Killing old processes...')
subprocess.run(['bash', '-c', 'pkill -f uvicorn 2>/dev/null; pkill -f ngrok 2>/dev/null; sleep 2'], capture_output=True)
time.sleep(2)

# STEP 2: Start backend FIRST and wait for it to be ready on localhost
print('Starting FastAPI backend...')
log_path = PROJECT_DIR + '/backend.log'

cmd = (
    'cd ' + PROJECT_DIR + ' && '
    'PYTHONPATH=' + PROJECT_DIR + ' '
    'HF_TOKEN=' + str(os.environ.get('HF_TOKEN', '')) + ' '
    'nohup python -m uvicorn app.main:app '
    '--host 0.0.0.0 --port 8000 '
    '--log-level info > ' + log_path + ' 2>&1 &'
)
subprocess.run(['bash', '-c', cmd], capture_output=True)
print('   Backend process started. Log: ' + log_path)
print()

# Wait for backend to be ready on localhost (IPv4 only)
print('Waiting for backend on localhost:8000...')
backend_ready = False
for i in range(30):
    time.sleep(5)
    try:
        resp = urllib.request.urlopen('http://127.0.0.1:8000/health', timeout=5)
        data = json.loads(resp.read())
        print('   Backend ready after ~' + str((i + 1) * 5) + 's')
        print('   GPU: ' + str(data.get('gpu_available')))
        backend_ready = True
        break
    except Exception:
        pass
    if i % 4 == 3:
        print('   Still waiting... (' + str((i + 1) * 5) + 's)')

if not backend_ready:
    print()
    print('!!! Backend not ready. Checking log:')
    try:
        with open(log_path, 'r') as f:
            lines = f.readlines()
        for line in lines[-30:]:
            sys.stdout.write(line.rstrip() + '\n')
            sys.stdout.flush()
    except Exception:
        sys.stdout.write('(log empty)\n')
    sys.stdout.flush()
    raise Exception('Backend failed to start. See log above.')

print()

# STEP 3: Now open ngrok tunnel (backend is already listening)
print('Opening ngrok tunnel...')
try:
    ngrok.kill()
except Exception:
    pass
time.sleep(1)

# Force IPv4 so ngrok connects to 127.0.0.1 not [::1]
http_tunnel = ngrok.connect(8000, bind_tls=True, host_header='127.0.0.1:8000')
public_url = http_tunnel.public_url
api_url = public_url + '/api'

print()
print('=' * 65)
print('BACKEND PUBLIC URL:')
print('   ' + public_url)
print('   API Base: ' + api_url)
print('=' * 65)
print()

# STEP 4: Final health check via public URL
print('Final health check via ngrok...')
try:
    resp = urllib.request.urlopen(public_url + '/health', timeout=10)
    data = json.loads(resp.read())
    print('SERVER READY!')
    print('   Backend: ' + str(data.get('service')))
    print('   GPU: ' + str(data.get('model', {}).get('gpu_available')))
except Exception as e:
    print('Warning: health via ngrok failed: ' + str(e)[:80])

print()
print('Backend log (last 30 lines):')
print('-' * 60)
try:
    with open(log_path, 'r') as f:
        lines = f.readlines()
    for line in lines[-30:]:
        sys.stdout.write(line.rstrip() + '\n')
        sys.stdout.flush()
except Exception:
    sys.stdout.write('(log empty)\n')
print('-' * 60)


---

### CELL 7: Theo dõi log (CHẠY SONG SONG với cell 6)

In [ ]:
!tail -60 /kaggle/working/AI_GEN_IMAGE/backend.log

---

### CELL 8: Test endpoints

In [ ]:
import urllib.request
import json
import re

PROJECT_DIR = '/kaggle/working/AI_GEN_IMAGE'
log_path = f'{PROJECT_DIR}/backend.log'

public_url = None
try:
    with open(log_path, 'r') as f:
        log = f.read()
    match = re.search(r'https://[\S]+\.ngrok[-_]?free[\S]*', log)
    if match:
        public_url = match.group().rstrip('.,')
except:
    pass

if not public_url:
    print('❌ Chưa có ngrok URL — chạy cell 6 trước')
else:
    print(f'🔗 Backend: {public_url}')
    print()
    for ep, label in [
        ('/health', 'Health'),
        ('/api/model/status', 'Model Status'),
        ('/api/lora', 'LoRA'),
        ('/api/export/formats', 'Export Formats'),
    ]:
        try:
            r = urllib.request.urlopen(f'{public_url}{ep}', timeout=10)
            d = json.loads(r.read())
            print(f'✅ {label}: OK')
        except Exception as e:
            print(f'❌ {label}: {str(e)[:60]}')
    print()
    print('📋 Log tail:')
    !tail -20 /kaggle/working/AI_GEN_IMAGE/backend.log

In [ ]:
import os
import gc
import time
import torch

PROJECT_DIR = '/kaggle/working/AI_GEN_IMAGE'
os.environ['PYTHONPATH'] = PROJECT_DIR

import sys
if PROJECT_DIR not in sys.path:
    sys.path.insert(0, PROJECT_DIR)

os.environ['LORA_OUTPUTS_DIR'] = PROJECT_DIR + '/outputs'
print('LORA_OUTPUTS_DIR=' + os.environ['LORA_OUTPUTS_DIR'])

HF_TOKEN = os.environ.get('HF_TOKEN', '')

t0 = time.time()

try:
    # Reset lora_manager singleton (modules + global variable)
    for mod in list(sys.modules.keys()):
        if 'lora_manager' in mod or mod.startswith('app.'):
            del sys.modules[mod]
    # Reset global singleton so it re-scans with fresh _discovered dict
    import app.services.lora_manager as _lm
    _lm._lora_manager = None
    print('Singleton + module cache reset')

    from diffusers import StableDiffusionPipeline, DPMSolverMultistepScheduler
    from app.services.lora_manager import get_lora_manager

    # Detect device BEFORE loading pipeline
    device = 'cuda' if torch.cuda.is_available() else 'cpu'
    dtype = torch.float16 if device == 'cuda' else torch.float32
    print('Device: {} | dtype: {}'.format(device, dtype))

    # Load base pipeline
    print('Downloading SDXL Turbo (~6GB)...')
    pipeline = StableDiffusionPipeline.from_pretrained(
        'stabilityai/sdxl-turbo',
        token=HF_TOKEN,
        variant='fp16' if device == 'cuda' else None,
        torch_dtype=dtype,
        safety_checker=None,
        requires_safety_checker=False,
    )
    print('Base model loaded')

    # Move pipeline to device
    pipeline = pipeline.to(device)
    print('Pipeline on {}'.format(device))

    # Load LoRA adapters
    lora_mgr = get_lora_manager()
    discovered = lora_mgr.available_types()
    print('LoRA Manager: {}'.format(discovered))
    if not discovered:
        print('   No LoRA found - using pure SDXL')
        print('   To use LoRA: upload outputs/ folder to Kaggle')
    else:
        for lora_type in discovered:
            ok = lora_mgr.load_adapter(lora_type, pipeline, scale=1.0, stack=False)
            status = 'OK' if ok else 'FAIL'
            print('   {} {}'.format(status, lora_type))
        if discovered:
            lora_mgr.unload_all(pipeline)

    # Apply scheduler + xformers
    if device == 'cuda':
        try:
            pipeline.enable_xformers_memory_efficient_attention()
            print('xformers enabled')
        except Exception as e:
            print('(xformers skip: {})'.format(e))
        pipeline.scheduler = DPMSolverMultistepScheduler.from_config(pipeline.scheduler.config)

    # Warmup
    print()
    print('Warmup GPU...')
    if device == 'cuda':
        torch.cuda.synchronize()
        print('GPU warmup done')
    print('Base model ready')

    # Cleanup
    del pipeline
    gc.collect()
    if device == 'cuda':
        torch.cuda.empty_cache()

    elapsed = time.time() - t0
    print()
    print('=' * 60)
    print('PRE-DOWNLOAD DONE! Time: {:.1f}s'.format(elapsed))
    print('   Base model cached - Cell 6 will start FAST')
    print('=' * 60)

except Exception as e:
    print('Error: {}'.format(e))
    import traceback
    traceback.print_exc()


In [ ]:
# ⭐ CELL 6: Start Backend + ngrok (FIXED)

import os
import subprocess
import time
import urllib.request
import json
from pyngrok import ngrok

PROJECT_DIR = '/kaggle/working/AI_GEN_IMAGE'
os.chdir(PROJECT_DIR)
os.environ['PYTHONPATH'] = PROJECT_DIR

# STEP 0: Remove any temp Python files to avoid uvicorn reload loop
subprocess.run(
    ['bash', '-c', 'rm -f ' + PROJECT_DIR + '/_ngrok_cell.py ' + PROJECT_DIR + '/_export_*.py ' + PROJECT_DIR + '/kaggle_cell*.py'],
    capture_output=True
)

# STEP 1: Kill old processes
print('Killing old processes...')
subprocess.run(['bash', '-c', 'pkill -f uvicorn 2>/dev/null; pkill -f ngrok 2>/dev/null; sleep 2'], capture_output=True)
time.sleep(2)

# STEP 2: Start backend FIRST and wait for it to be ready on localhost
print('Starting FastAPI backend...')
log_path = PROJECT_DIR + '/backend.log'

cmd = (
    'cd ' + PROJECT_DIR + ' && '
    'PYTHONPATH=' + PROJECT_DIR + ' '
    'HF_TOKEN=' + str(os.environ.get('HF_TOKEN', '')) + ' '
    'nohup python -m uvicorn app.main:app '
    '--host 0.0.0.0 --port 8000 '
    '--log-level info > ' + log_path + ' 2>&1 &'
)
subprocess.run(['bash', '-c', cmd], capture_output=True)
print('   Backend process started. Log: ' + log_path)
print()

# Wait for backend to be ready on localhost (IPv4 only)
print('Waiting for backend on localhost:8000...')
backend_ready = False
for i in range(30):
    time.sleep(5)
    try:
        resp = urllib.request.urlopen('http://127.0.0.1:8000/health', timeout=5)
        data = json.loads(resp.read())
        print('   Backend ready after ~' + str((i + 1) * 5) + 's')
        print('   GPU: ' + str(data.get('gpu_available')))
        backend_ready = True
        break
    except Exception:
        pass
    if i % 4 == 3:
        print('   Still waiting... (' + str((i + 1) * 5) + 's)')

if not backend_ready:
    print()
    print('!!! Backend not ready. Checking log:')
    try:
        with open(log_path, 'r') as f:
            lines = f.readlines()
        for line in lines[-30:]:
            sys.stdout.write(line.rstrip() + '\n')
            sys.stdout.flush()
    except Exception:
        sys.stdout.write('(log empty)\n')
    sys.stdout.flush()
    raise Exception('Backend failed to start. See log above.')

print()

# STEP 3: Now open ngrok tunnel (backend is already listening)
print('Opening ngrok tunnel...')
try:
    ngrok.kill()
except Exception:
    pass
time.sleep(1)

# Force IPv4 so ngrok connects to 127.0.0.1 not [::1]
http_tunnel = ngrok.connect(8000, bind_tls=True, host_header='127.0.0.1:8000')
public_url = http_tunnel.public_url
api_url = public_url + '/api'

print()
print('=' * 65)
print('BACKEND PUBLIC URL:')
print('   ' + public_url)
print('   API Base: ' + api_url)
print('=' * 65)
print()

# STEP 4: Final health check via public URL
print('Final health check via ngrok...')
try:
    resp = urllib.request.urlopen(public_url + '/health', timeout=10)
    data = json.loads(resp.read())
    print('SERVER READY!')
    print('   Backend: ' + str(data.get('service')))
    print('   GPU: ' + str(data.get('model', {}).get('gpu_available')))
except Exception as e:
    print('Warning: health via ngrok failed: ' + str(e)[:80])

print()
print('Backend log (last 30 lines):')
print('-' * 60)
try:
    with open(log_path, 'r') as f:
        lines = f.readlines()
    for line in lines[-30:]:
        sys.stdout.write(line.rstrip() + '\n')
        sys.stdout.flush()
except Exception:
    sys.stdout.write('(log empty)\n')
print('-' * 60)


In [ ]:
# Test các endpoint
import urllib.request
import json
import re
import os

PROJECT_DIR = '/kaggle/working/AI_GEN_IMAGE'
log_path = f'{PROJECT_DIR}/backend.log'

# Tìm ngrok URL trong log
public_url = None
try:
    with open(log_path, 'r') as f:
        log = f.read()
    match = re.search(r'https://[\S]+\.ngrok[-_]?free[\S]*', log)
    if match:
        public_url = match.group().rstrip('.,')
except:
    pass

if not public_url:
    print('❌ Chưa có ngrok URL — chạy cell 6 trước')
else:
    api_url = public_url + '/api'
    print(f'🔗 Backend: {public_url}')
    print()
    for ep, label in [
        ('/health', 'Health'),
        ('/api/model/status', 'Model Status'),
        ('/api/lora', 'LoRA'),
        ('/api/export/formats', 'Export Formats'),
    ]:
        try:
            r = urllib.request.urlopen(f'{public_url}{ep}', timeout=10)
            d = json.loads(r.read())
            print(f'✅ {label}: OK')
        except Exception as e:
            print(f'❌ {label}: {str(e)[:60]}')
    print()
    print('📋 Log tail:')
    !tail -20 /kaggle/working/AI_GEN_IMAGE/backend.log

---

## 🔧 Debug

```bash
!cat /kaggle/working/AI_GEN_IMAGE/backend.log
!ps aux | grep uvicorn
```